# Spam Email Classifier

Detecting spam emails using NLP — TF-IDF feature extraction + Naive Bayes and Logistic Regression.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')
print('Ready!')

## Step 1 — Generate and explore the dataset

In [ ]:
import subprocess
subprocess.run(['python', '../generate_dataset.py'])
df = pd.read_csv('../data/emails.csv')
print(df['label'].value_counts())
df.head()

## Step 2 — Text Preprocessing

In [ ]:
import sys
sys.path.append('..')
from preprocessor import preprocess_batch

df['clean_text'] = preprocess_batch(df['text'].tolist())
print('Original:', df['text'].iloc[0])
print('Cleaned :', df['clean_text'].iloc[0])

## Step 3 — Feature Extraction (BoW vs TF-IDF)

In [ ]:
X = df['clean_text']
y = (df['label'] == 'spam').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
print(f'TF-IDF matrix shape: {X_train_tfidf.shape}')

## Step 4 — Train Naive Bayes

In [ ]:
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train_tfidf, y_train)
preds_nb = nb.predict(X_test_tfidf)
print('Naive Bayes:')
print(classification_report(y_test, preds_nb, target_names=['Ham','Spam']))

## Step 5 — Train Logistic Regression

In [ ]:
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
preds_lr = lr.predict(X_test_tfidf)
print('Logistic Regression:')
print(classification_report(y_test, preds_lr, target_names=['Ham','Spam']))

## Step 6 — Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10,4))
for ax,(name,preds) in zip(axes,[('Naive Bayes',preds_nb),('Logistic Regression',preds_lr)]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Ham','Spam'], yticklabels=['Ham','Spam'])
    ax.set_title(name)
plt.tight_layout()
plt.show()

## Step 7 — Top Spam/Ham words

In [ ]:
features = tfidf.get_feature_names_out()
coef     = lr.coef_[0]

fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax, idx, title, color in [
    (axes[0], np.argsort(coef)[-15:], 'Top Spam words', '#E24B4A'),
    (axes[1], np.argsort(coef)[:15],  'Top Ham words',  '#1D9E75')
]:
    ax.barh(features[idx], np.abs(coef[idx]), color=color)
    ax.set_title(title, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 8 — Test on a new email

In [ ]:
from preprocessor import preprocess

test_email = 'CONGRATULATIONS! You have won a FREE iPhone. Click here to claim your prize now!'
cleaned    = preprocess(test_email)
features_v = tfidf.transform([cleaned])
prob       = lr.predict_proba(features_v)[0]

print(f'Email   : {test_email}')
print(f'Verdict : {"SPAM" if prob[1]>0.5 else "HAM"}')
print(f'Spam probability: {prob[1]*100:.1f}%')